# Chest X-Ray Pneumonia Detection with CNN (Transfer Learning)

## Objectives
Binary image classification: **Normal** vs **Pneumonia** using CNN + MobileNetV2 backbone.

## CNN + Transfer Learning
**Transfer learning** reuses weights trained on ImageNet. We freeze early layers and train only the top **classifier head**, saving data and training time.

## Note
Download the [Chest X-Ray Pneumonia](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia) dataset and extract to `chest_xray_data/`. A fallback demo uses CIFAR10 if folders are missing.


In [ ]:
# Optional: install dependencies (uncomment if needed)
# !pip install -q numpy pandas matplotlib seaborn scikit-learn tensorflow requests yfinance

import warnings
warnings.filterwarnings("ignore")

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    mean_squared_error, mean_absolute_error, r2_score,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_theme(style="whitegrid")
print("TensorFlow:", tf.__version__)


## Problem Definition & Business Context
Hospitals use imaging AI to **triage** chest X-rays. Models assist radiologists; they require clinical validation, bias audits, and regulatory approval before production use.


In [ ]:
import zipfile
import requests
from pathlib import Path

# Small public sample — user may replace with full Kaggle download
# Using tensorflow_demo style: we build from directory structure after manual download OR use cats/dogs style subset
# Stable approach: use keras TSV + folder from TFDS alternative — here we use a documented folder layout

DATA_DIR = Path("chest_xray_data")
if not (DATA_DIR / "train").exists():
    print("Download chest x-ray zip from Kaggle and extract to chest_xray_data/train and test")
    print("Expected: chest_xray_data/train/NORMAL, chest_xray_data/train/PNEUMONIA, ...")
else:
    print("Found local chest xray folders")

IMG_SIZE = (160, 160)
BATCH = 32

if (DATA_DIR / "train").exists():
    train_ds = keras.utils.image_dataset_from_directory(
        DATA_DIR / "train", image_size=IMG_SIZE, batch_size=BATCH, seed=SEED, validation_split=0.2, subset="training"
    )
    val_ds = keras.utils.image_dataset_from_directory(
        DATA_DIR / "train", image_size=IMG_SIZE, batch_size=BATCH, seed=SEED, validation_split=0.2, subset="validation"
    )
    test_ds = keras.utils.image_dataset_from_directory(DATA_DIR / "test", image_size=IMG_SIZE, batch_size=BATCH)
    class_names = train_ds.class_names
    print("Classes:", class_names)
else:
    # Fallback demo pipeline with CIFAR grayscale simulation (documented)
    print("Using CIFAR10 subset as DEMO pipeline when chest data missing — replace with real x-ray folders.")
    (x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
    y_train, y_test = y_train.squeeze(), y_test.squeeze()
    binary = (y_train < 2).astype(int), (y_test < 2).astype(int)
    train_ds = tf.data.Dataset.from_tensor_slices((x_train[:5000], binary[0][:5000])).batch(BATCH)
    val_ds = tf.data.Dataset.from_tensor_slices((x_train[5000:6000], binary[0][5000:6000])).batch(BATCH)
    test_ds = tf.data.Dataset.from_tensor_slices((x_test[:1000], binary[1][:1000])).batch(BATCH)


In [ ]:
# EDA — class balance and sample images
if 'class_names' in dir():
    counts = {c: 0 for c in class_names}
    for _, labels in train_ds:
        for lab in labels.numpy():
            counts[class_names[int(lab)]] += 1
        break
    print("Sample batch class counts:", counts)

for images, labels in train_ds.take(1):
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    for ax, img, lab in zip(axes.ravel(), images[:8], labels[:8]):
        ax.imshow((img.numpy() * 255).astype("uint8") if img.numpy().max() <= 1 else img.numpy().astype("uint8"))
        ax.set_title(class_names[int(lab)] if 'class_names' in dir() else str(int(lab)))
        ax.axis("off")
    plt.suptitle("Sample X-ray / demo images")
    plt.show()


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def prep(img, label):
    img = tf.image.resize(img, (160, 160))
    img = tf.cast(img, tf.float32)
    return keras.applications.mobilenet_v2.preprocess_input(img), label

train_batched = train_ds.map(prep, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_batched = val_ds.map(prep, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_batched = test_ds.map(prep, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)


In [ ]:
base = keras.applications.MobileNetV2(include_top=False, weights="imagenet", input_shape=(160, 160, 3))
base.trainable = False
inputs = layers.Input(shape=(160, 160, 3))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)
model.compile(optimizer=keras.optimizers.Adam(1e-4), loss="binary_crossentropy", metrics=["accuracy", "AUC"])
model.summary()


In [ ]:
xray_cb = [
    callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor="val_auc", mode="max"),
    callbacks.ModelCheckpoint("cnn_xray_best.keras", save_best_only=True, monitor="val_auc", mode="max"),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3),
]
history = model.fit(train_batched, validation_data=val_batched, epochs=15, callbacks=xray_cb, verbose=1)
pd.DataFrame(history.history).plot(figsize=(10,4))
plt.title("Transfer learning training curves")
plt.show()


In [ ]:
loss, acc, auc = model.evaluate(test_batched, verbose=0)
print(f"Test loss={loss:.4f} acc={acc:.4f} auc={auc:.4f}")

y_true, y_prob = [], []
for imgs, labs in test_batched:
    p = model.predict(imgs, verbose=0).ravel()
    y_prob.extend(p)
    y_true.extend(labs.numpy())
y_pred = (np.array(y_prob) >= 0.5).astype(int)
print(classification_report(y_true, y_pred))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens")
plt.title("Pneumonia detection — confusion matrix")
plt.show()


In [ ]:
# Inference demo
for imgs, labs in test_batched.take(1):
    p = model.predict(imgs[:3], verbose=0).ravel()
    for i, prob in enumerate(p):
        print(f"Image {i}: P(pneumonia)={prob:.3f} pred={int(prob>=0.5)} true={int(labs[i])}")
model.save("cnn_xray_deploy.keras")


## Deployment Notes

1. **Serving**: Export with `model.export("saved_model")` for TensorFlow Serving, or wrap `predict` in FastAPI/Flask.
2. **Preprocessing**: Always apply the **same** scaler/encoder fitted on training data (`scaler.pkl`).
3. **Monitoring**: Track input drift, latency, and prediction distribution on live traffic.
4. **Retraining**: Schedule periodic retrain when performance drops below SLA.
5. **Security**: Do not log PII; use HTTPS and auth on inference endpoints.
